# CalmFruits — неделя 3: единственная финальная оценка

Эта тетрадь запускается только после фиксации `reports/selected_configuration.json` и review экспериментального протокола. Первый успешный запуск читает официальный test ровно один раз; последующие запуски читают сохранённый итоговый результат.

In [ ]:
from __future__ import annotations

import importlib.metadata
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from joblib import load
from scipy import sparse

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
assert (ROOT / 'src' / 'calmfruits').exists()
sys.path.insert(0, str(ROOT / 'src'))

from calmfruits.data import load_local_env, read_parquet_from_s3
from calmfruits.final import assert_final_input_contract, run_final_once
from calmfruits.provenance import write_manifest
from calmfruits.search import LexicalSearch
from calmfruits.tracking import log_evaluation_run

CATALOG_DIR = ROOT / 'artifacts' / 'catalog'
INDEX_DIR = ROOT / 'artifacts' / 'indexes'
FINAL_DIR = ROOT / 'artifacts' / 'final'
REPORT_DIR = ROOT / 'reports'
assert load_local_env(ROOT / '.env')
environment = pd.DataFrame({'value': {'python':sys.version.split()[0], 'pandas':importlib.metadata.version('pandas'), 'mlflow':importlib.metadata.version('mlflow'), 'device':'CPU', 'purpose':'one-time final test evaluation'}})
display(environment)


## 1. Проверка замороженной конфигурации

До чтения test подтверждаем, что validation-победитель, каталог и поисковые артефакты существуют и согласованы.

In [ ]:
selected = json.loads((REPORT_DIR / 'selected_configuration.json').read_text())
assert selected['winner']['method'] == 'lexical', 'The frozen winner must be loaded by its declared method'
catalog = pd.read_parquet(CATALOG_DIR / 'evaluation_catalog.parquet')
lexical = LexicalSearch(catalog, load(INDEX_DIR / 'tfidf_vectorizer.joblib'), sparse.load_npz(INDEX_DIR / 'tfidf_matrix.npz'))
assert catalog.imt_id.is_unique and len(catalog) == selected['winner'].get('catalog_size', len(catalog))
frozen_selection = {'method': selected['winner']['method'], 'validation_run_id': selected['run_ids'][selected['winner']['method']], 'fingerprints': selected['fingerprints']}
write_manifest(FINAL_DIR / 'frozen_selection.json', frozen_selection)
display(pd.DataFrame({'value': frozen_selection}))


**Вывод по `frozen_selection`.** Финальная конфигурация зафиксирована до чтения test; результаты будут сформулированы после единственного завершённого прогона.

## 2. Одноразовый test-прогон

При первом запуске загружается test, проверяется отсутствие пересечения с train и вычисляется только замороженный TF-IDF. Каждая обработанная выдача добавляется в checkpoint; completed state предотвращает повторный прогон.

In [ ]:
state_path = FINAL_DIR / 'final_test_state.json'
if state_path.exists() and json.loads(state_path.read_text())['status'] == 'completed':
    final_per_query = pd.read_csv(FINAL_DIR / 'per_query_metrics.csv')
    final_top10 = pd.read_csv(FINAL_DIR / 'top10_results.csv')
    final_summary = pd.read_csv(FINAL_DIR / 'summary.csv')
    final_test_mode = 'loaded_completed_checkpoint'
else:
    train = read_parquet_from_s3('queries_synthetic_train.parquet')
    test = read_parquet_from_s3('queries_synthetic_test.parquet')
    assert_final_input_contract(test, train)
    final_per_query, final_top10, final_summary = run_final_once(lexical.search_lexical, test, catalog.imt_id, frozen_selection, FINAL_DIR)
    final_test_mode = 'executed_once'
final_summary.to_csv(REPORT_DIR / 'final_test_metrics.csv', index=False)
display(pd.DataFrame({'mode':[final_test_mode], 'test_queries':[final_per_query.query_id.nunique()], 'catalog_size':[len(catalog)]}))
display(final_summary)


**Вывод по `final_summary`.** Финальные метрики и их интерпретация будут добавлены только после завершённого test-прогона.

## 3. Финальные ошибки и рекомендации

Показываем три test-запроса с наименьшим NDCG@10 и сопоставляем выдачу с размеченными товарами.

In [ ]:
worst_test_queries = final_per_query.nsmallest(3, 'ndcg_at_10')[['query_id','query_text','ndcg_at_10','mrr','reachable_relevant_share']]
worst_test_top10 = final_top10.merge(worst_test_queries[['query_id']], on='query_id', how='inner')
display(worst_test_queries)
display(worst_test_top10[['query_id','rank','imt_id','relevance','imt_name','subj_name','score']])


**Вывод по `worst_test_queries` и `worst_test_top10`.** Анализ ограничений, рекомендация бизнесу и дальнейшие шаги будут добавлены только после завершённого test-прогона.

## 4. MLflow и проверка сохранения

Логируем финальную test-оценку замороженного победителя, сохраняя результаты и manifest как артефакты.

In [ ]:
final_run_id_path = FINAL_DIR / 'mlflow_run_id.txt'
if final_run_id_path.exists():
    final_run_id = final_run_id_path.read_text().strip()
else:
    final_run_id = log_evaluation_run('lexical_final_test', {'method':'lexical','eval_split':'test','catalog_size':len(catalog),'relevance_threshold':2,'validation_run_id':frozen_selection['validation_run_id'],'selection_fingerprints':json.dumps(frozen_selection['fingerprints'], sort_keys=True)}, final_summary, [FINAL_DIR / 'per_query_metrics.csv', FINAL_DIR / 'summary.csv', FINAL_DIR / 'frozen_selection.json'])
    final_run_id_path.write_text(final_run_id + '\n')
display(pd.DataFrame({'final_mlflow_run_id':[final_run_id], 'final_state':[json.loads(state_path.read_text())['status']]}))


**Вывод по final MLflow run.** Test run ID и финальные выводы будут добавлены только после завершённого прогона.